# Módulo 08 · Aula 01 — Fundamentos de Containers

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Contratamos uma desenvolvedora nova. Ela levou **dois dias** para conseguir rodar o Atlas. Python errado, PostgreSQL de outra versão, o Mongo não subia no Windows, e uma variável de ambiente que ninguém lembrava de documentar."*

E, uma semana depois:

> *"O relatório funciona na sua máquina e quebra na minha. Mesmo código, mesmo commit."*

O Atlas hoje precisa de: Python 3.10+, PostgreSQL 16, MongoDB 7, Redis, cinco pacotes do sistema e nove variáveis de ambiente. Cada pessoa monta isso à mão, do seu jeito, no seu sistema operacional.

> 🎯 **"Na minha máquina funciona" não é desculpa — é um diagnóstico.**
>
> Ele diz que o ambiente é uma variável não controlada do sistema. E toda variável não controlada acaba aparecendo em produção, no pior momento.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | O problema do ambiente | Por que containers existem |
| 2 | **O que um container É** | 🎯 Ao vivo, sem Docker, com `unshare` |
| 3 | Os cinco namespaces | A isolação de verdade |
| 4 | cgroups | O limite de recursos |
| 5 | Container vs VM | E quando cada um serve |
| 6 | Imagem vs container | A confusão nº 1 de iniciante |
| 7 | Comandos essenciais | `run`, `ps`, `logs`, `exec`, `stop` |
| 8 | Ciclo de vida | E onde os dados somem |

> 💭 **Este módulo é diferente dos anteriores.** Docker é uma ferramenta de linha de comando que exige um *daemon* — e ele não roda dentro de todo ambiente. Leia a próxima célula com atenção: ela explica o que é executado de verdade e o que é referência.

## ⚙️ Como este notebook funciona

Duas coisas acontecem aqui, e é importante você saber qual é qual:

| | O que é | Como aparece |
|---|---------|--------------|
| ✅ **Executado** | Namespaces do Linux via `unshare`, análise de arquivos, cgroups | Saída normal |
| 📖 **Referência** | Comandos `docker ...` | Marcados com `[referência]` |

**Por quê?** Porque um daemon Docker não roda dentro de todo ambiente — nem dentro de um container, nem em CI restrito. Em vez de fingir, o notebook é honesto.

> 🎯 **E há um ganho nisso.** Em vez de decorar `docker run`, você vai **construir um container à mão** com as chamadas de sistema que o Docker usa por baixo. Depois disso, `docker run` deixa de ser mágica.
>
> 🐳 **Se você tiver Docker instalado e rodando**, a célula abaixo detecta e os comandos executam de verdade.

⚠️ Os demos de namespace exigem **Linux**. No Windows, rode-os dentro do WSL2 — que, aliás, é o que o Docker Desktop usa por baixo.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 08
# ═══════════════════════════════════════════════════════════════
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

E_LINUX = platform.system() == "Linux"
TEM_DOCKER = shutil.which("docker") is not None
DOCKER_LIGADO = False
if TEM_DOCKER:
    DOCKER_LIGADO = subprocess.run(["docker", "info"], capture_output=True).returncode == 0


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            return False


_garantir("pyyaml", "yaml")
import yaml

print(f"sistema : {platform.system()}")
print(f"docker  : {'🐳 disponível e ligado' if DOCKER_LIGADO else ('instalado, mas o daemon não responde' if TEM_DOCKER else 'não instalado')}")


# ═══════════════════════════════════════════════════════════════
#  Executar comandos
# ═══════════════════════════════════════════════════════════════

def sh(comando: str, mostrar: bool = True, cwd=None, timeout: int = 120) -> str:
    """Roda um comando de shell e devolve a saída."""
    processo = subprocess.run(comando, shell=True, capture_output=True,
                              text=True, cwd=cwd, timeout=timeout)
    saida = (processo.stdout + processo.stderr).rstrip()
    if mostrar and saida:
        print(saida)
    return saida


def docker(comando: str, esperado: str | None = None, mostrar: bool = True) -> str:
    """Executa `docker ...` se houver daemon; senão, mostra o comando.

    💭 Por que este modo duplo?

       Um daemon Docker não roda dentro de todo ambiente (nem dentro de
       um container, nem em CI restrito, nem neste avaliador). Em vez de
       fingir que rodou, o notebook é HONESTO: quando não há daemon, ele
       mostra o comando e a saída típica, marcada como referência.

       🔴 Saída marcada `[referência]` NÃO foi executada. Rode você
          mesmo no terminal — é assim que se aprende Docker.
    """
    linha = f"docker {comando}"
    if DOCKER_LIGADO:
        print(f"$ {linha}")
        return sh(linha, mostrar=mostrar)
    print(f"$ {linha}")
    if esperado:
        for l in textwrap.dedent(esperado).strip("\n").splitlines():
            print(f"  {l}")
    print("  ── [referência] o daemon Docker não está disponível aqui ──")
    return esperado or ""


# ═══════════════════════════════════════════════════════════════
#  Pasta de trabalho
# ═══════════════════════════════════════════════════════════════
def preparar(nome: str) -> Path:
    base = Path(nome).resolve()
    if base.exists():
        shutil.rmtree(base)
    base.mkdir(parents=True)
    return base


def arvore(raiz: Path, prefixo: str = ""):
    itens = [i for i in sorted(raiz.iterdir(), key=lambda p: (p.is_file(), p.name))
             if i.name not in {"__pycache__", ".git"}]
    for i, item in enumerate(itens):
        ultimo = i == len(itens) - 1
        marca = "└── " if ultimo else "├── "
        tamanho = f"  ({item.stat().st_size} B)" if item.is_file() else ""
        print(f"{prefixo}{marca}{item.name}{tamanho}")
        if item.is_dir():
            arvore(item, prefixo + ("    " if ultimo else "│   "))


def tabela(cabecalho: list[str], linhas: list[list], larguras: list[int]):
    """Tabela ASCII alinhada.

    ⚠️ Marcadores ASCII, não emoji: `len("⚠️")` é 2 mas o terminal
       desenha 1 coluna, e a tabela sai torta. Você já viu isso no M03.
    """
    print("  ".join(f"{c:<{w}}" for c, w in zip(cabecalho, larguras)))
    print("─" * (sum(larguras) + 2 * len(larguras)))
    for linha in linhas:
        print("  ".join(f"{str(c):<{w}}" for c, w in zip(linha, larguras)))


print("✅ `sh()`, `docker()`, `preparar()`, `arvore()` e `tabela()` prontos")

## 1. O problema

Antes da solução, meça a dor.

In [ ]:
# O que o Atlas precisa hoje, na máquina de cada pessoa
DEPENDENCIAS = [
    ("Python",      "3.10+",  "sistema", "match/case, int | None"),
    ("PostgreSQL",  "16",     "serviço", "MVCC, JSONB, timestamptz"),
    ("MongoDB",     "7",      "serviço", "catálogo de estrutura variável"),
    ("Redis",       "7",      "serviço", "cache e idempotência"),
    ("libpq-dev",   "—",      "sistema", "compilar o psycopg"),
    ("gcc",         "—",      "sistema", "compilar extensões C"),
]
VARIAVEIS = ["ATLAS_DB_URL", "ATLAS_MONGO_URI", "ATLAS_REDIS_URL",
             "ATLAS_SECRET_KEY", "ATLAS_AMBIENTE", "ATLAS_TOKEN_EXPIRA_MINUTOS",
             "ATLAS_ORIGENS_PERMITIDAS", "VELOZ_CLIENTE_ID", "VELOZ_SEGREDO"]

tabela(["COMPONENTE", "VERSÃO", "TIPO", "POR QUÊ"],
       DEPENDENCIAS, [14, 8, 9, 34])

print(f"\n{len(DEPENDENCIAS)} componentes + {len(VARIAVEIS)} variáveis de ambiente")
print(f"× 3 sistemas operacionais (Windows, macOS, Linux)")
print(f"× N pessoas na equipe\n")
print("🔴 Cada combinação é uma chance de 'na minha máquina funciona'.")

In [ ]:
# O ambiente ONDE ESTE NOTEBOOK RODA — e por que ele é diferente do seu
print(f"sistema      : {platform.system()} {platform.release()}")
print(f"python       : {sys.version.split()[0]}")
print(f"arquitetura  : {platform.machine()}")
print(f"núcleos      : {os.cpu_count()}")

for pacote in ["fastapi", "sqlalchemy", "httpx", "pydantic"]:
    try:
        import importlib.metadata as md
        print(f"  {pacote:<12} {md.version(pacote)}")
    except Exception:
        print(f"  {pacote:<12} ausente")

print("\n💭 Anote esses números. Se você rodar o Atlas numa máquina com")
print("   Python 3.9, metade da sintaxe do M04 quebra — e o erro vai")
print("   apontar para o SEU código, não para a versão errada.")

> 💭 **As três não-soluções que todo mundo tenta antes:**
>
> | Tentativa | Por que falha |
> |-----------|---------------|
> | Um `README` com os passos | Fica desatualizado na primeira semana |
> | Um script `setup.sh` | Funciona em um sistema operacional só |
> | *"Use as mesmas versões que eu"* | Ninguém tem as mesmas versões de tudo |
>
> O `venv` (M01) resolveu o problema **das bibliotecas Python**. Ele não resolve PostgreSQL, Mongo, Redis, `gcc` nem a versão do próprio Python.
>
> 🎯 **Container é o `venv` do sistema operacional inteiro.**

## 2. 🎯 O que um container É

Aqui está a parte que a maioria dos cursos pula.

Um container **não é** uma máquina virtual leve. Não é uma caixa. É um **processo comum do Linux** — que enxerga menos do que existe.

O truque tem três partes:

| Mecanismo | O que faz | Analogia |
|-----------|-----------|----------|
| **Namespaces** | Limitam o que o processo **vê** | Janela estreita |
| **cgroups** | Limitam o que o processo **usa** | Cota de consumo |
| **Sistema de arquivos em camadas** | Dá a ele um "/" próprio | Cenário de teatro |

Vamos provar isso — **sem Docker**, com o comando `unshare`, que expõe as mesmas chamadas de sistema.

In [ ]:
# Este notebook consegue rodar os demos?
POSSIVEL = E_LINUX and shutil.which("unshare") is not None
if POSSIVEL:
    teste = subprocess.run("unshare --user --map-root-user true",
                           shell=True, capture_output=True)
    POSSIVEL = teste.returncode == 0

print(f"Linux          : {E_LINUX}")
print(f"unshare        : {shutil.which('unshare') or 'ausente'}")
print(f"namespaces     : {'✅ podemos demonstrar ao vivo' if POSSIVEL else '⚠️ indisponível aqui'}")

if not POSSIVEL:
    print("\n💡 No Windows: abra o WSL2 e rode este notebook de lá.")
    print("   No macOS: os namespaces são do kernel Linux; use uma VM ou")
    print("   leia as saídas de referência abaixo.")

In [ ]:
# ═══ NAMESPACE 1: USER — quem você é ═══
if POSSIVEL:
    print(f"FORA   : uid={os.getuid()}  ({sh('id -un', mostrar=False)})")
    print(sh("unshare --user --map-root-user sh -c "
             "'echo \"DENTRO : uid=$(id -u)  ($(id -un))\"'", mostrar=False))
else:
    print("FORA   : uid=1000  (fabio)")
    print("DENTRO : uid=0     (root)          [referência]")

print("\n🎯 O MESMO processo, dois valores de uid.")
print("   Ninguém virou administrador da máquina: o kernel está")
print("   MENTINDO para o processo, de propósito e com segurança.")

> 🎯 **Isto é o `USER namespace`, e é a base da segurança de containers.**
>
> Dentro do container, o processo se acha `root` — pode instalar pacote, escrever em `/etc`, mudar permissões. Fora, ele continua sendo o seu usuário comum, sem poder nenhum sobre o host.
>
> 🔴 **Mas atenção:** isso vale para *rootless containers* e para o `unshare` acima. No Docker padrão (com daemon rodando como root), `root` dentro do container é **de verdade** `root` no host se ele escapar. É por isso que "não rode o processo como root dentro da imagem" é uma regra real — voltamos a ela na aula 08_02.

In [ ]:
# ═══ NAMESPACE 2: PID — quais processos existem ═══
if POSSIVEL:
    fora = sh("ps -e --no-headers | wc -l", mostrar=False).strip()
    dentro = sh("unshare --user --map-root-user --pid --fork --mount-proc "
                "sh -c 'ps -e --no-headers | wc -l'", mostrar=False).strip()
    print(f"FORA   : {fora} processos visíveis")
    print(f"DENTRO : {dentro} processos visíveis")
    print()
    print("Tabela de processos DENTRO do namespace:")
    print(sh("unshare --user --map-root-user --pid --fork --mount-proc "
             "ps -eo pid,comm", mostrar=False))
else:
    print("FORA   : 312 processos visíveis")
    print("DENTRO : 2 processos visíveis        [referência]")
    print("  PID COMMAND")
    print("    1 ps")

print("\n🎯 Dentro, o SEU processo é o PID 1.")
print("   Ele não enxerga — e não consegue matar — nada do host.")
print("\n💡 Se o número de 'FORA' parecer baixo, é porque este notebook")
print("   já roda num ambiente isolado. Na sua máquina, rode")
print("   `ps -e | wc -l` num terminal: são centenas.")

> 💭 **O PID 1 tem uma responsabilidade especial**, e isso te morde mais tarde.
>
> No Linux, o PID 1 é o `init`: ele adota processos órfãos e precisa tratar sinais. Um processo comum promovido a PID 1 **não trata `SIGTERM` por padrão** — e é por isso que `docker stop` num container mal construído demora 10 segundos e termina em `SIGKILL`.
>
> Você vai ver esse problema — e a correção — na aula 08_02.

In [ ]:
# ═══ NAMESPACE 3: NET — quais interfaces de rede existem ═══
if POSSIVEL:
    print("FORA:")
    print(sh("ip -o link show | awk '{print \"   \"$2}'", mostrar=False))
    print("DENTRO (namespace de rede novo):")
    print(sh("unshare --user --map-root-user --net "
             "ip -o link show | awk '{print \"   \"$2}'", mostrar=False))
else:
    print("FORA:\n   lo:\n   eth0:\n   docker0:")
    print("DENTRO:\n   lo:                    [referência]")

print("\n🎯 Uma pilha de rede INTEIRA e própria: interfaces, rotas,")
print("   tabelas de firewall, portas.")
print("\n💡 É por isso que dois containers podem ambos escutar na porta")
print("   8000 sem conflito — são portas 8000 de PILHAS DIFERENTES.")
print("   E é por isso que existe o `-p 8080:8000`: alguém precisa")
print("   ligar as duas pilhas.")

In [ ]:
# ═══ NAMESPACE 4: MOUNT — qual sistema de arquivos existe ═══
if POSSIVEL:
    saida = sh("""unshare --user --map-root-user --mount sh -c '
        mkdir -p /tmp/dentro_do_container
        mount -t tmpfs tmpfs /tmp/dentro_do_container
        echo "senha-do-banco" > /tmp/dentro_do_container/segredo.txt
        echo "   DENTRO vê: $(ls /tmp/dentro_do_container)"
        echo "   conteúdo : $(cat /tmp/dentro_do_container/segredo.txt)"
    '""", mostrar=False)
    print(saida)
    fora = sh("ls /tmp/dentro_do_container 2>/dev/null", mostrar=False)
    print(f"   FORA vê  : [{fora or 'nada — o diretório está vazio'}]")
else:
    print("   DENTRO vê: segredo.txt")
    print("   conteúdo : senha-do-banco")
    print("   FORA vê  : [nada]              [referência]")

print("\n🎯 O arquivo existiu, foi escrito, foi lido — e desapareceu")
print("   junto com o processo. Fora do namespace ele NUNCA existiu.")
print("\n💭 Guarde isso: é exatamente por aqui que os dados de um")
print("   container somem. Voltaremos a isso na seção 8.")

In [ ]:
# ═══ NAMESPACE 5: UTS — qual é o nome da máquina ═══
if POSSIVEL:
    print(f"FORA   : hostname = {platform.node()}")
    print(sh("unshare --user --map-root-user --uts sh -c "
             "'hostname atlas-api-7f3d9 && echo \"DENTRO : hostname = $(hostname)\"'",
             mostrar=False))
else:
    print("FORA   : hostname = notebook-fabio")
    print("DENTRO : hostname = atlas-api-7f3d9    [referência]")

print("\n💡 É daqui que vem aquele hostname esquisito quando você faz")
print("   `docker exec -it ... bash`: é o id do container.")

In [ ]:
# ═══ TODOS JUNTOS: um container feito à mão ═══
if POSSIVEL:
    print("Criando um 'container' com os cinco namespaces:\n")
    print(sh("""unshare --user --map-root-user --pid --fork --mount-proc \
                --net --uts --ipc sh -c '
        hostname atlas-container
        echo "  hostname : $(hostname)"
        echo "  usuário  : $(id -un) (uid $(id -u))"
        echo "  processos: $(ps -e --no-headers | wc -l)"
        echo "  interfaces: $(ip -o link show | wc -l)"
        echo "  PID atual : $$"
    '""", mostrar=False))
else:
    print("  hostname : atlas-container")
    print("  usuário  : root (uid 0)")
    print("  processos: 2")
    print("  interfaces: 1")
    print("  PID atual : 1                  [referência]")

print("\n" + "═" * 58)
print("🎯 ISTO É UM CONTAINER.")
print("═" * 58)
print("""
Nenhuma máquina virtual. Nenhum hipervisor. Nenhum kernel novo.

Um processo comum, rodando no MESMO kernel do host, com cinco
chamadas ao sistema dizendo "você enxerga menos".

O Docker faz exatamente isto — e mais três coisas:
  · monta um sistema de arquivos em camadas como "/"
  · aplica limites de CPU e memória com cgroups
  · gerencia rede, volumes, imagens e o ciclo de vida
""")

> 🎯 **Depois desta seção, `docker run` deixa de ser mágica.**
>
> Quando você digitar `docker run -it ubuntu bash` e aparecer um prompt de root numa máquina Ubuntu, você sabe o que aconteceu: o Docker criou os namespaces, montou uma árvore de arquivos Ubuntu como `/`, e executou `bash` lá dentro. O kernel continua sendo o seu.
>
> 💭 **E isso explica a limitação mais importante do Docker:** como o kernel é compartilhado, **um container Linux precisa de um kernel Linux**. Docker no Windows e no macOS roda uma VM Linux discreta por baixo — o Docker Desktop é, em boa parte, um gerenciador dessa VM.

## 3. cgroups — o limite de consumo

Namespaces limitam o que o processo **vê**. Eles não limitam o que ele **usa**: um processo isolado ainda pode consumir 100% da CPU e toda a memória.

Quem faz isso são os **cgroups** (*control groups*).

In [ ]:
# Este ambiente está sob cgroups?
CGROUP = Path("/sys/fs/cgroup")
if CGROUP.exists():
    print("Controladores disponíveis:")
    ctrl = (CGROUP / "cgroup.controllers")
    print(f"   {ctrl.read_text().strip() if ctrl.exists() else 'v1 (layout antigo)'}\n")

    for arquivo, rotulo in [("memory.max", "limite de memória"),
                            ("memory.current", "memória em uso"),
                            ("cpu.max", "limite de CPU"),
                            ("pids.max", "limite de processos")]:
        p = CGROUP / arquivo
        if p.exists():
            valor = p.read_text().strip()
            if valor == "max":
                valor = "max (sem limite)"
            elif arquivo == "memory.current":
                valor = f"{int(valor) / 1024**2:,.0f} MB"
            elif arquivo == "memory.max" and valor.isdigit():
                valor = f"{int(valor) / 1024**2:,.0f} MB"
            print(f"   {rotulo:<22} {valor}")
else:
    print("⚠️ /sys/fs/cgroup indisponível (não é Linux, ou está oculto)")

print(f"\n   núcleos visíveis       {os.cpu_count()}")

> ⚠️ **A armadilha que derruba aplicação Java e Python em produção:**
>
> `os.cpu_count()` costuma devolver os núcleos **do host**, não os do container. Uma aplicação que cria "um worker por CPU" vê 64 núcleos, cria 64 workers — e o cgroup lhe deu 0,5 CPU.
>
> O resultado é troca de contexto sem fim e uma aplicação lentíssima **sem nenhum erro no log**.
>
> 🧭 **A correção:** leia o limite de verdade, ou declare o número de workers explicitamente. Voltamos a isso no M09, ao configurar o Gunicorn.

In [ ]:
# Como se limita um container (referência)
docker("run --memory=512m --cpus=0.5 --pids-limit=100 atlas-api", esperado="""
# memória máxima: 512 MB — estourou, o kernel MATA o processo (OOMKilled)
# CPU: metade de um núcleo
# no máximo 100 processos (contra fork bomb)
""")

print()
docker("stats --no-stream", esperado="""
CONTAINER ID   NAME        CPU %   MEM USAGE / LIMIT   MEM %   PIDS
a1b2c3d4e5f6   atlas-api   0.42%   87.4MiB / 512MiB    17.07%  9
7f8e9d0c1b2a   atlas-db    1.13%   142.9MiB / 1GiB     13.96%  23
""")

print("\n🔴 `OOMKilled` é o erro mais confuso do Docker: o container morre")
print("   sem mensagem no log da aplicação. Não foi bug — foi o kernel")
print("   cumprindo o limite que VOCÊ definiu.")
print("   Investigue com: docker inspect <id> | grep -i oom")

## 4. Container vs máquina virtual

In [ ]:
comparacao = [
    ["Kernel",          "compartilhado com o host", "próprio, completo"],
    ["Tempo de partida", "milissegundos",            "dezenas de segundos"],
    ["Tamanho típico",   "50 MB - 500 MB",           "1 GB - 20 GB"],
    ["Overhead de CPU",  "praticamente zero",        "5% - 15%"],
    ["Densidade",        "centenas por host",        "dezenas por host"],
    ["Isolamento",       "bom (kernel compartilhado)", "forte (hardware)"],
    ["Outro SO",         "não (só o mesmo kernel)",  "sim (Windows sobre Linux)"],
]
tabela(["", "CONTAINER", "MÁQUINA VIRTUAL"], comparacao, [18, 27, 26])

print("""
💭 QUANDO CADA UM

   Container   empacotar e distribuir aplicação; microsserviços;
               CI; ambiente reproduzível de desenvolvimento

   VM          rodar outro sistema operacional; isolamento forte
               entre clientes diferentes (multi-tenant); kernel
               customizado; conformidade que exige separação física

   Na prática  containers RODAM DENTRO de VMs na nuvem. Não é
               "ou um ou outro" — são camadas diferentes.
""")

> 🔴 **"Isolamento bom" não é "isolamento forte".**
>
> Como o kernel é compartilhado, uma vulnerabilidade de *escape* no kernel afeta todos os containers da máquina. Uma VM comprometida não alcança as outras da mesma forma.
>
> **Consequência prática:** não confie só no container para isolar código não confiável de clientes diferentes. Para isso existem VMs, gVisor e Firecracker.

## 5. Imagem vs container — a confusão nº 1

In [ ]:
print("""
   IMAGEM                          CONTAINER
   ══════                          ═════════
   Molde, receita, classe          Instância, execução, objeto
   Somente leitura                 Tem uma camada gravável por cima
   Versionada por tag              Efêmera
   Vive num registro               Vive no seu daemon
   `docker build` cria             `docker run` cria
   `docker images` lista           `docker ps` lista

   Uma imagem  →  zero, um ou MIL containers ao mesmo tempo
""")

print("💭 A analogia mais próxima do que você já sabe (M04):\n")
print("   imagem    ≈  a classe `Produto`")
print("   container ≈  o objeto `Produto(sku='NB-DELL-15')`")
print("\n   Você não 'roda uma classe'. Você instancia.")

In [ ]:
docker("images", esperado="""
REPOSITORY    TAG       IMAGE ID       CREATED         SIZE
atlas-api     1.2.0     3f2a1b8c9d0e   2 hours ago     187MB
atlas-api     latest    3f2a1b8c9d0e   2 hours ago     187MB
postgres      16        a1b2c3d4e5f6   3 weeks ago     432MB
redis         7-alpine  9f8e7d6c5b4a   1 month ago     41MB
""")

print()
docker("ps -a", esperado="""
CONTAINER ID   IMAGE           COMMAND        STATUS                    PORTS      NAMES
a1b2c3d4e5f6   atlas-api:1.2.0 "uvicorn ..."  Up 2 hours                0.0.0.0:8000->8000/tcp  atlas-api
7f8e9d0c1b2a   postgres:16     "postgres"     Up 2 hours (healthy)      5432/tcp   atlas-db
2b3c4d5e6f7a   atlas-api:1.2.0 "python -m ..." Exited (0) 5 minutes ago            atlas-migracao
""")

print("\n💡 Repare no terceiro: `Exited (0)` é SUCESSO — era uma tarefa")
print("   pontual (rodar as migrações), não um serviço. Nem todo")
print("   container fica no ar; alguns fazem uma coisa e terminam.")

> ⚠️ **`latest` não significa "a mais nova".**
>
> É só uma tag como qualquer outra — a tag padrão quando você não especifica nenhuma. Uma imagem publicada há dois anos pode estar marcada como `latest`.
>
> 🔴 **E usar `latest` em produção é a receita para o build de hoje ser diferente do de ontem**, sem que nada no seu código tenha mudado. Fixe a versão. Sempre.

## 6. Os comandos que você vai usar todo dia

In [ ]:
comandos = [
    ["docker run <img>",          "cria E inicia um container"],
    ["docker run -d <img>",       "em segundo plano (detached)"],
    ["docker run -it <img> bash", "interativo, com terminal"],
    ["docker run -p 8080:8000",   "porta do host : porta do container"],
    ["docker run -e VAR=valor",   "variável de ambiente"],
    ["docker run --rm",           "💡 remove ao terminar"],
    ["docker run --name atlas",   "nome legível em vez de hash"],
    ["", ""],
    ["docker ps",                 "containers RODANDO"],
    ["docker ps -a",              "⚠️ TODOS, inclusive parados"],
    ["docker logs <c>",           "saída padrão do container"],
    ["docker logs -f <c>",        "acompanha em tempo real"],
    ["docker exec -it <c> bash",  "🔧 abre um shell DENTRO"],
    ["docker inspect <c>",        "tudo sobre o container, em JSON"],
    ["", ""],
    ["docker stop <c>",           "SIGTERM, espera 10s, SIGKILL"],
    ["docker kill <c>",           "🔴 SIGKILL direto"],
    ["docker rm <c>",             "remove um container parado"],
    ["docker rmi <img>",          "remove uma imagem"],
    ["docker system prune -a",    "🔴 apaga TUDO que não está em uso"],
]
for cmd, desc in comandos:
    print(f"   {cmd:<30}{desc}" if cmd else "")

In [ ]:
# O primeiro container
docker("run --rm hello-world", esperado="""
Unable to find image 'hello-world:latest' locally
latest: Pulling from library/hello-world
c1ec31eb5944: Pull complete
Digest: sha256:d211f485f2dd1dee407a80973c8f129f00d54604d2c90732e8e320e5038a0348
Status: Downloaded newer image for hello-world:latest

Hello from Docker!
This message shows that your installation appears to be working correctly.
""")

print("\n💡 Repare em 'Unable to find image ... locally' seguido de")
print("   'Pulling'. O `run` baixa a imagem se ela não existir.")
print("   E o `--rm` fez o container sumir assim que terminou.")

In [ ]:
# Rodando o Atlas (referência)
docker("""run -d \\
    --name atlas-api \\
    -p 8000:8000 \\
    -e ATLAS_AMBIENTE=desenvolvimento \\
    -e ATLAS_SECRET_KEY=$ATLAS_SECRET_KEY \\
    --memory=512m --cpus=1.0 \\
    --restart unless-stopped \\
    atlas-api:1.2.0""", esperado="""
a1b2c3d4e5f67890abcdef1234567890abcdef1234567890abcdef1234567890
""")

print()
docker("logs --tail 5 atlas-api", esperado="""
   [lifespan] tabelas prontas, API no ar
INFO:     Started server process [1]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000
""")

print("\n🔴 REPARE: `--host 0.0.0.0`, não `127.0.0.1`.")
print("   Dentro do container, `127.0.0.1` é o loopback DAQUELE")
print("   namespace de rede. Um servidor escutando lá é inalcançável")
print("   de fora, mesmo com `-p 8000:8000` — e o erro é um")
print("   'connection refused' sem nenhuma pista.")
print("\n   É o erro nº 1 de quem containeriza a primeira aplicação.")

> 🔧 **`-p 8080:8000` — qual número é qual?**
>
> ```
> -p <porta do HOST>:<porta do CONTAINER>
>      ↑ o que você acessa    ↑ onde a aplicação escuta
> ```
>
> `-p 8080:8000` significa: *"quem bater na porta 8080 da minha máquina, mande para a porta 8000 do container"*. Você abre `http://localhost:8080`.
>
> 💡 **Como lembrar:** é a mesma ordem de `docker cp` e de volumes — **de fora para dentro**, sempre.

## 7. 🔴 Onde os dados somem

Esta é a lição mais cara do módulo.

In [ ]:
print("""
CICLO DE VIDA

   docker run          created ──► running
   docker pause                      │  └──► paused
   docker stop                       ▼
                                  exited          ← 🔴 os dados AINDA existem
   docker rm                          │
                                      ▼
                                   removido       ← 🔴 os dados SUMIRAM
""")

print("""
🔴 A CAMADA GRAVÁVEL É DO CONTAINER, NÃO DA IMAGEM.

   Tudo que o container escreve — arquivos, o banco de dados, uploads —
   vive numa camada temporária que é criada com ele e DESTRUÍDA com ele.

   `docker stop`  →  os dados continuam lá (o container só parou)
   `docker rm`    →  os dados vão junto, sem confirmação
   `--rm`         →  🔴 os dados somem assim que o processo terminar
""")

print("💭 O incidente clássico:\n")
print("   1. sobe o Postgres com `docker run -d postgres`")
print("   2. usa por três semanas em desenvolvimento")
print("   3. precisa mudar uma variável de ambiente")
print("   4. `docker rm atlas-db && docker run -d ...`")
print("   5. 🔴 três semanas de dados de teste, e o schema, sumiram")
print("\n   Nenhum aviso. Nenhuma confirmação. O Docker fez o que você pediu.")

In [ ]:
# A solução: volumes (detalhada na aula 08_02)
docker("volume create atlas-dados", esperado="atlas-dados")
print()
docker("run -d --name atlas-db -v atlas-dados:/var/lib/postgresql/data postgres:16",
       esperado="7f8e9d0c1b2a3456789...")
print()
docker("volume ls", esperado="""
DRIVER    VOLUME NAME
local     atlas-dados
""")

print("\n✅ Agora `docker rm atlas-db` NÃO apaga os dados.")
print("   O volume é um objeto separado, com ciclo de vida próprio.")
print("\n⚠️ E `docker volume rm atlas-dados` apaga. Sem confirmação.")

## 🔧 Prática guiada — inventário do que precisa ser containerizado

Antes de escrever o primeiro `Dockerfile` (próxima aula), vamos mapear o Atlas.

In [ ]:
BASE = preparar("aula_08_01")

# O que o Atlas é hoje, e o que cada peça vira
SERVICOS = [
    # nome,        imagem base,          porta,  volume?,  papel
    ("atlas-api",  "python:3.12-slim",   8000,   "não",    "a aplicação (você constrói)"),
    ("atlas-db",   "postgres:16",        5432,   "SIM",    "transacional (M05)"),
    ("atlas-mongo", "mongo:7",           27017,  "SIM",    "catálogo (M05)"),
    ("atlas-redis", "redis:7-alpine",    6379,   "talvez", "cache e idempotência (M07)"),
]
tabela(["SERVIÇO", "IMAGEM BASE", "PORTA", "VOLUME", "PAPEL"],
       [[n, i, p, v, d] for n, i, p, v, d in SERVICOS], [12, 20, 6, 8, 30])

print("""
💭 REPARE NA COLUNA "VOLUME" — ela é a pergunta mais importante:

   "se este container for destruído, alguma coisa se perde?"

   atlas-db     SIM      🔴 os pedidos da Aurora
   atlas-mongo  SIM      🔴 o catálogo
   atlas-redis  talvez   cache pode ser reconstruído; idempotência não
   atlas-api    não      ✅ ela não guarda estado — é para ser assim
""")

print("🎯 Uma aplicação SEM ESTADO pode ser destruída e recriada à")
print("   vontade. É isso que torna possível escalar, atualizar sem")
print("   parar e se recuperar de falha automaticamente (M09).")

In [ ]:
# Onde o estado do Atlas vive hoje
estado = [
    ["dados/atlas.db",      "SQLite (M03)",     "🔴 vira volume ou Postgres"],
    ["saida/*.txt|json|csv", "relatórios",       "⚠️ volume ou objeto (S3)"],
    ["saida/atlas.jsonl",   "log estruturado",  "🔴 NÃO: log vai para stdout"],
    [".env",                "segredos",         "🔴 NUNCA na imagem"],
    ["dados/brutos/*.csv",  "entrada",          "bind mount ou volume"],
    ["__pycache__/",        "cache do Python",  "✅ .dockerignore"],
    [".venv/",              "ambiente virtual", "🔴 .dockerignore — é do SEU sistema"],
]
tabela(["CAMINHO", "O QUE É", "O QUE FAZER"], estado, [22, 18, 34])

print("""
🔴 A LINHA DO LOG É A MAIS IMPORTANTE.

   Aplicação em container NÃO escreve log em arquivo. Ela escreve em
   `stdout` e `stderr`, e o Docker recolhe.

   Por quê? Porque o arquivo de log vive na camada gravável — que morre
   com o container. Você perde exatamente o log de que precisa para
   entender por que o container morreu.

   💡 O `observabilidade.py` do M04 já escreve JSON. Só falta mandá-lo
      para o stdout em vez de `saida/atlas.jsonl`.
""")

In [ ]:
# Uma primeira lista de checagem, gravada no projeto
CHECAGEM = BASE / "PRE_CONTAINER.md"
CHECAGEM.write_text("""# Antes de containerizar o Atlas

## Configuração
- [ ] Toda configuração vem do ambiente (feito no M06: `ConfigAPI`)
- [ ] Nenhum caminho absoluto da minha máquina no código
- [ ] `.env` no `.gitignore` e fora da imagem

## Estado
- [ ] A API não guarda nada em disco local
- [ ] Banco e cache são serviços separados, não processos internos
- [ ] Uploads vão para volume ou armazenamento de objetos

## Log
- [ ] Log vai para stdout/stderr, não para arquivo
- [ ] Formato estruturado (JSON) — já feito no M04

## Processo
- [ ] A aplicação roda em primeiro plano (sem `&`, sem daemon)
- [ ] Ela trata SIGTERM e encerra com educação
- [ ] Existe uma rota de saúde (`/saude`, feita no M06)

## Rede
- [ ] O servidor escuta em `0.0.0.0`, não `127.0.0.1`
- [ ] A porta é configurável por variável de ambiente
""", encoding="utf-8")

print(f"✅ {CHECAGEM.name} criado\n")
print(CHECAGEM.read_text(encoding="utf-8"))

In [ ]:
# Quanto do Atlas JÁ está pronto graças aos módulos anteriores
prontos = [
    ["Configuração por ambiente", "M06", "✅ ConfigAPI com BaseSettings"],
    ["Log estruturado",           "M04", "🔶 falta mandar para stdout"],
    ["Rota de saúde",             "M06", "✅ GET /saude"],
    ["Sem estado na API",         "M06", "✅ sessão por requisição"],
    ["Segredos fora do código",   "M06", "✅ .env + .gitignore"],
    ["Dependências declaradas",   "M04", "✅ pyproject.toml"],
    ["Escuta em 0.0.0.0",         "—",   "🔴 hoje é 127.0.0.1"],
    ["Trata SIGTERM",             "—",   "🔴 falta"],
]
tabela(["REQUISITO", "VEIO DO", "SITUAÇÃO"], prontos, [28, 8, 34])

print("\n💭 Seis dos oito requisitos já estavam prontos — e você não os")
print("   fez pensando em Docker. Fez porque eram boas práticas.")
print("\n🎯 Containerizar não é adaptar a aplicação ao Docker.")
print("   É perceber que uma aplicação bem construída JÁ É containerizável.")
print("\n   O que sobra são dois ajustes de dez minutos. É esse o retorno")
print("   dos módulos anteriores.")

## 📝 Exercícios

**E1.** Rode `unshare --user --map-root-user id` e explique, num comentário, por que você "virou root" sem ter dado nenhuma senha.

**E2.** Crie um namespace de PID e mostre a tabela de processos dentro dele. Explique por que o seu processo é o PID 1.

**E3.** 🔴 Monte um `tmpfs` dentro de um namespace de mount, escreva um arquivo, e prove que ele não existe fora. Relacione com a camada gravável do container.

**E4.** Crie um namespace de rede e liste as interfaces. Explique por que dois containers podem escutar na mesma porta.

**E5.** Combine os cinco namespaces num único comando e descreva o que você construiu.

**E6.** Leia `/sys/fs/cgroup/memory.max` e `cpu.max` do seu sistema. Se disserem `max`, explique o que isso significa.

**E7.** Instale o Docker e rode `docker run --rm hello-world`. Descreva cada linha da saída.

**E8.** 🔴 Suba um container com `--memory=64m` rodando um script Python que aloca 200 MB numa lista. Mostre o `OOMKilled` no `docker inspect`.

**E9.** Suba um `nginx` com `-p 8080:80`, acesse pelo navegador, e depois com `-p 8080:8080`. Explique por que o segundo não funciona.

**E10.** Rode um container com `-d`, use `docker exec -it ... sh`, crie um arquivo, saia, faça `docker restart` e verifique se o arquivo continua lá. Depois `docker rm` e recrie. Explique a diferença.

**E11.** Compare `docker stop` e `docker kill` num container que escreve no log a cada segundo. Meça o tempo de cada um.

**E12.** 🔴 Suba um Postgres sem volume, crie uma tabela, remova o container e recrie. Mostre que a tabela sumiu. Depois refaça com `-v`.

**E13.** Rode `docker inspect` num container e encontre: a camada gravável, os namespaces, os limites de cgroup e a configuração de rede.

**E14.** Percorra o `PRE_CONTAINER.md` desta aula com o SEU `projeto_Atlas` e marque o que já está pronto.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

## 📋 Cola de referência

```bash
# ═══ O que um container É ═══
# Um PROCESSO com visão restrita. Sem VM, sem kernel novo.
#   namespaces  limitam o que ele VÊ    (user, pid, net, mnt, uts, ipc)
#   cgroups     limitam o que ele USA   (cpu, memória, pids)
#   camadas     dão a ele um "/" próprio

unshare --user --map-root-user --pid --fork --mount-proc --net --uts sh

# ═══ Imagem × Container ═══
#   imagem    molde, só leitura, versionada    ≈ classe
#   container instância, camada gravável       ≈ objeto

# ═══ Rodar ═══
docker run <img>                  cria e inicia
  -d                              segundo plano
  -it                             interativo com terminal
  -p 8080:8000                    HOST:CONTAINER  (de fora para dentro)
  -e VAR=valor                    variável de ambiente
  --env-file .env                 várias de uma vez
  -v nome:/caminho                volume  🔴 sem isto, os dados somem
  -v $(pwd)/src:/app/src          bind mount (desenvolvimento)
  --rm                            remove ao terminar
  --name atlas                    nome legível
  --memory=512m --cpus=1.0        limites
  --restart unless-stopped        reinicia sozinho

# ═══ Inspecionar ═══
docker ps                 rodando       ·  docker ps -a       todos
docker logs -f <c>        acompanhar    ·  docker logs --tail 50
docker exec -it <c> bash  🔧 entrar     ·  docker inspect <c>
docker stats              consumo ao vivo
docker top <c>            processos do container

# ═══ Encerrar ═══
docker stop <c>           SIGTERM → 10s → SIGKILL
docker kill <c>           🔴 SIGKILL direto
docker rm <c>             🔴 remove o container E a camada gravável
docker system prune -a    🔴 apaga tudo que não está em uso

# ═══ 🔴 ARMADILHAS ═══
# 1. escute em 0.0.0.0, NUNCA 127.0.0.1 (senão -p não adianta)
# 2. sem volume, `docker rm` apaga os dados sem avisar
# 3. `latest` não é "a mais nova" — é só a tag padrão
# 4. log vai para stdout, não para arquivo
# 5. OOMKilled = o kernel cumpriu o limite que VOCÊ pôs
# 6. os.cpu_count() mente dentro do container
```

## ✅ Checklist de saída

- [ ] Sei explicar por que "na minha máquina funciona" é um diagnóstico
- [ ] 🎯 **Sei que um container é um processo, não uma VM**
- [ ] Nomeio os cinco namespaces e o que cada um isola
- [ ] Sei a diferença entre namespaces (ver) e cgroups (usar)
- [ ] Sei por que um container Linux exige um kernel Linux
- [ ] Distingo imagem de container
- [ ] Sei que `latest` não significa "a mais nova"
- [ ] 🔴 **Sei que a aplicação precisa escutar em `0.0.0.0`**
- [ ] Sei a ordem de `-p HOST:CONTAINER`
- [ ] 🔴 **Sei que `docker rm` apaga os dados da camada gravável**
- [ ] Sei que log vai para stdout, não para arquivo
- [ ] Reconheço `OOMKilled` e sei o que causou
- [ ] Sei por que o PID 1 precisa tratar `SIGTERM`
- [ ] Percorri o `PRE_CONTAINER.md` com o meu projeto

---

### ➡️ Próxima aula

**`08_02_Criando_Imagens.ipynb`** — O `Dockerfile`: camadas, cache, multi-stage, `.dockerignore` e as três regras de segurança. Você vai construir um analisador que encontra os problemas antes do `docker build`.